In [1]:
import pandas as pd
import xml.etree.ElementTree as ET

from IPython.display import display, HTML

PATH_ACTIONS                        = "../../Data/Unprocessed/actions.xml"
PATH_ACTIONS_FILTERED               = "../../Data/Processed/actions_filtered.csv"
PATH_ROI                            = "../../Data/Unprocessed/roi.csv"
PATH_LABEL_MAP                      = "../../Data/label_map.csv"

MINIMUM_DURATION                    = 4
MINIMUM_ACTION_COUNT                = 5

In [2]:
def parse_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    records = []

    # --------------------------------------------------------------------------
    # STEP 1: Extract task-level metadata (id → source filename)
    # --------------------------------------------------------------------------
    task_sources = {}
    for task in root.findall(".//meta/project/tasks/task"):
        task_id = task.findtext("id")
        source = task.findtext("source")
        name = task.findtext("name")

        if task_id:
            task_sources[task_id] = {
                "source": source,
                "name": name
            }

    # --------------------------------------------------------------------------
    # STEP 2: Parse all tracks and link to the correct task
    # --------------------------------------------------------------------------
    for track in root.findall("track"):
        task_id = track.get("task_id")
        label = track.get("label")

        # Match this track to its source video via task_id
        task_info = task_sources.get(task_id, {})
        source = task_info.get("source", "unknown")
        task_name = task_info.get("name", "unknown")

        # ---- Extract per-frame boxes and attributes ----
        for box in track.findall("box"):
            frame = int(box.get("frame"))

            if label == "Strip":
                xtl = float(box.get("xtl"))
                ytl = float(box.get("ytl"))
                xbr = float(box.get("xbr"))
                ybr = float(box.get("ybr"))

                frame_attrs = {
                    attr.get("name"): attr.text.strip() if attr.text else ""
                    for attr in box.findall("attribute")
                }

                records.append({
                    "task_name": task_name,
                    "source": source,
                    "frame": frame,
                    "xtl": xtl,
                    "ytl": ytl,
                    "xbr": xbr,
                    "ybr": ybr,
                    **frame_attrs
                })

    return pd.DataFrame(records)

In [3]:
def combine_frames(df):
    df["clip_number"] = df["file"].str.extract(r"^(\d+)").astype(int)
    df["frame"] = df.groupby(["file"])["frame"].transform(lambda x: x - x.min())

    # Sort by hierarchy including numeric clip number
    df = df.sort_values(["file", "fencer", "frame"]).reset_index(drop=True)

    # Columns to group by for hierarchy
    group_cols = ["file", "fencer"]
    results = []

    # Iterate over groups
    for _, grp in df.groupby(group_cols):
        grp = grp.sort_values("frame").reset_index(drop=True)
        
        # Use shift/cumsum to identify consecutive runs
        grp["run"] = (grp["action"] != grp["action"].shift()).cumsum()
        
        # Aggregate start/end frames per run
        run_df = grp.groupby(["run", "action"]).agg(
            start_frame=("frame", "min"),
            end_frame=("frame", "max")
        ).reset_index(drop=False)
        
        # Add hierarchy columns
        run_df["file"] = grp["file"].iloc[0]
        run_df["fencer"] = grp["fencer"].iloc[0]
        
        # Keep only desired columns
        run_df = run_df[["file", "fencer", "action", "start_frame", "end_frame"]]
        results.append(run_df)

    # Combine all groups
    return pd.concat(results, ignore_index=True)

In [4]:
def parse_annotations(path_annotations):
    df = parse_xml(path_annotations)

    df["task_name"] = df["task_name"].str.replace("Bout ", "", regex=False)
    df["task_name"] = df["task_name"].replace("Test Upload", "1")
    df["file"] = df["task_name"] + "/" + df["source"]

    df_roi = df[["file", "frame", "xtl", "ytl", "xbr", "ybr"]]
    df_roi.to_csv(PATH_ROI)

    df = df.melt(
        id_vars=["file", "frame"],
        value_vars=["Fencer_L", "Fencer_R"],
        var_name="fencer",
        value_name="action"
    )

    df["fencer"] = df["fencer"].map({
        "Fencer_L": "LEFT",
        "Fencer_R": "RIGHT"
    })

    cols = ["file", "frame", "fencer", "action"]
    df = df[cols]

    return combine_frames(df[cols])

In [5]:
def show_stats(df):
    def display_actions(df):
        html_str = ""
        title = "<h4>Actions</h4>"
        html_str += f"""
        <div style="display: inline-block; vertical-align: top; margin-right: 30px;">
            {title}
            {df.to_html(index=False)}
        </div>
        """
        display(HTML(html_str))

    def get_stats(df):
        df = df.copy()
        df["duration"] = df["end_frame"] - df["start_frame"] + 1
        stats = (
            df.groupby("action")["duration"]
            .agg(["min", "max", "mean", "std"])
            .reset_index()
        )

        stats["count"] = df["action"].value_counts().reindex(stats["action"]).values
        return stats.sort_values(by="count", ascending=False)

    total_actions = len(df)
    stats = get_stats(df)

    display_actions(stats)

    print(f"\nTotal Frames: {total_actions}")
    print("Total classes: ", df["action"].nunique())
    print("")

In [6]:
df = parse_annotations(PATH_ACTIONS)

df.sort_values(by=["file", "fencer", "start_frame"], inplace=True)
df = df.reset_index(drop=True)

df["duration"] = df["end_frame"] - df["start_frame"] + 1
df["action_id"] = df.index.astype(int)

df = df[["file", "fencer", "action_id", "action", "start_frame", "end_frame"]]

show_stats(df)
print("")
print(df)

df.to_csv(PATH_ACTIONS_FILTERED, index=False)

action,min,max,mean,std,count
NO_ACTION,2,159,19.495784,18.041536,593
LONG_ATTACK,2,17,7.374468,2.240885,235
SHORT_ATTACK,3,82,11.646409,14.497005,181
DIST_PULL,3,34,13.141304,7.659773,92
PARRY,3,20,8.346154,3.687192,26



Total Frames: 1127
Total classes:  5


               file fencer  action_id        action  start_frame  end_frame
0     1/10_Left.mp4   LEFT          0     NO_ACTION            0         22
1     1/10_Left.mp4   LEFT          1   LONG_ATTACK           23         27
2     1/10_Left.mp4  RIGHT          2     NO_ACTION            0         20
3     1/10_Left.mp4  RIGHT          3   LONG_ATTACK           21         27
4     1/11_Left.mp4   LEFT          4     NO_ACTION            0         59
...             ...    ...        ...           ...          ...        ...
1122   6/9_Left.mp4   LEFT       1122     DIST_PULL           36         44
1123   6/9_Left.mp4  RIGHT       1123     NO_ACTION            0         19
1124   6/9_Left.mp4  RIGHT       1124   LONG_ATTACK           20         27
1125   6/9_Left.mp4  RIGHT       1125     NO_ACTION           28         38
1126   6/9_Left.mp4  RIGHT       1126  SHORT_ATTACK           39         44

[1127 rows x 6 columns]


In [9]:
unique_labels = sorted(df["action"].unique())

labels = pd.DataFrame(unique_labels, columns=["action"])
labels["id"] = labels.index

labels.rename(columns={"action": "label", "action_id": "weight"}, inplace=True)
labels.to_csv(PATH_LABEL_MAP, index=False)

print(labels)

          label  id
0     DIST_PULL   0
1   LONG_ATTACK   1
2     NO_ACTION   2
3         PARRY   3
4  SHORT_ATTACK   4
